# D532 GCP Python: Work with GCS

Use Python to get the configured project ID, create a bucket, list buckets, upload a file, list files, download a file, and delete the uploaded file.

Use the existing Ubuntu **dataengenv** environment. Run Python cells in order after completing the Console setup. The bucket and object operations use real Google Cloud resources.

A light AWS mapping: `boto3` provides AWS clients; **google-cloud-storage** provides the GCS client.

## 1. Open the notebook with dataengenv

Run these commands in the Ubuntu terminal. Adjust the activation path if your environment is elsewhere.

```bash
source ~/dataengenv/bin/activate
python -m pip install ipykernel
python -m ipykernel install --user --name dataengenv --display-name "Python (dataengenv)"
```

Open this notebook in Jupyter or VS Code and select **Python (dataengenv)** as its kernel.

Install the storage library using the next notebook cell. `%pip` installs into the selected kernel environment. `google-auth` is included explicitly because the authentication example imports it.

[Cloud Storage Python library](https://docs.cloud.google.com/storage/docs/reference/libraries)

In [ ]:
%pip install google-cloud-storage google-auth

## 2. Obtain the project ID in Google Cloud Console

1. Open [Google Cloud Console](https://console.cloud.google.com/).
2. Click the **project selector** beside the Google Cloud logo.
3. Select the project you will use. If you need a new one and have permission, choose **New Project**, enter a name, choose an organization or location if requested, and click **Create**.
4. Open **IAM & Admin ? Settings** for the selected project and copy its **Project ID**. You can also find it in the project selector or the dashboard's project information.
5. Open **Billing** and link an available billing account if the project is not already linked.
6. Open **APIs & Services ? Library**, search for **Cloud Storage API**, and enable it if necessary.

| Value | Example | Use here |
|---|---|---|
| Project name | Retail Data Engineering | Display label |
| Project ID | `retail-de-example` | Pass to the Python client |
| Project number | A numeric identifier | Not used in these examples |

The project ID identifies the destination project; it is not a password or an authentication credential.

[Project documentation](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects)

## 3. Choose the authentication method

For the Console-based credential example, use a **service account JSON key**. The next sections explain how to create and use it. If ADC was configured in D531, you can choose that alternative in the authentication cell instead.

| AWS idea | GCP approach used here |
|---|---|
| Access key ID and secret access key | Service account identity and private key held in a JSON file |
| SDK uses credentials to sign requests | Google libraries obtain access tokens using the credentials |
| IAM policies grant access | IAM roles grant access to the service account or signed-in user |

There is no AWS-style access-key/secret pair to paste into this GCS client. Do not create a generic **API key** under APIs & Services for these private-bucket operations. GCS HMAC keys are for interoperability workflows and are not used by this notebook.

## 4. Create a service account in the Console

1. Select the intended project.
2. Open **IAM & Admin ? Service Accounts**.
3. Click **Create service account**.
4. Enter a name such as `gcs-python-demo` and a description such as `Python GCS exercises`.
5. Click **Create and continue**.
6. In the project access step, select **Cloud Storage ? Storage Admin** (`roles/storage.admin`) for this dedicated exercise project.
7. Continue and click **Done**. The optional user-access step can remain empty for this example.

The account email looks like `gcs-python-demo@YOUR_PROJECT_ID.iam.gserviceaccount.com`.

Storage Admin covers bucket creation and listing plus object upload, download, listing, and deletion. It is broad; use it in a dedicated exercise project. Object-only roles do not grant bucket creation.

If the service account already exists, use **IAM & Admin ? IAM ? Grant access**, enter its email, and assign the role there.

Creating accounts requires appropriate service-account creation permissions; granting project roles requires project IAM policy permissions. If those controls are unavailable, have the project administrator perform these steps.

[Create service accounts](https://docs.cloud.google.com/iam/docs/service-accounts-create) ? [Storage roles](https://docs.cloud.google.com/storage/docs/access-control/iam-roles)

## 5. Download the JSON credential file

1. Open **IAM & Admin ? Service Accounts**.
2. Click the service account you created.
3. Open **Keys ? Add key ? Create new key**.
4. Select **JSON**, then click **Create**. Save the downloaded file.
5. Place it on the Ubuntu machine running the notebook, outside the notebook folder or Git repository.

The JSON contains fields such as `project_id`, `client_email`, and `private_key`. Keep the file private; do not paste its contents into a notebook.

Example Ubuntu terminal commands after moving the downloaded file to Ubuntu's Downloads folder; replace the source filename:

```bash
mkdir -p ~/.config/gcp
mv ~/Downloads/YOUR_DOWNLOADED_KEY.json ~/.config/gcp/gcs-python-demo.json
chmod 600 ~/.config/gcp/gcs-python-demo.json
```

Key creation requires appropriate permissions, such as **Service Account Key Admin**. An organization policy may prohibit keys. In that case, use the ADC option below rather than changing the policy.

When this credential is no longer needed, remove it from the service account's **Keys** tab. Deleting only the local file does not revoke the key.

[Service account keys](https://docs.cloud.google.com/iam/docs/keys-create-delete)

## 6. Alternative: use the ADC login from D531

For local development, ADC lets Python use your signed-in account without downloading a service account key. Skip this section if using the JSON key.

Run in an Ubuntu terminal as the same OS user who runs Jupyter:

```bash
gcloud auth application-default login
```

For a remote machine without a usable browser:

```bash
gcloud auth application-default login --no-launch-browser
```

The Google account selected during login needs the storage permissions from Section 4. CLI login with `gcloud auth login` alone does not configure Python ADC.

In the next cell, select `AUTH_METHOD = "adc"`. For this user-login route, leave `GOOGLE_APPLICATION_CREDENTIALS` unset; otherwise ADC may load that file instead.

[Local ADC setup](https://docs.cloud.google.com/docs/authentication/set-up-adc-local-dev-environment)

## 7. Create the Python client and get the project ID

Set `PROJECT_ID` to the value copied from the Console. Select `service_account` or `adc`. For the JSON method, set `KEY_PATH` to the file on the machine running the kernel.

The JSON's project ID is the service account's home project. `client.project` is the target project selected here; these can differ when cross-project permissions are granted. This cell reads the configured project identity, not a list of all projects.

In [ ]:
from pathlib import Path
import google.auth
from google.cloud import storage
from google.oauth2 import service_account

AUTH_METHOD = "service_account"  # Change to "adc" to use the D531 login.
PROJECT_ID = "YOUR_PROJECT_ID"
KEY_PATH = Path("~/.config/gcp/gcs-python-demo.json").expanduser()

if AUTH_METHOD == "service_account":
    credentials = service_account.Credentials.from_service_account_file(str(KEY_PATH))
    credential_project = credentials.project_id
elif AUTH_METHOD == "adc":
    credentials, credential_project = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
else:
    raise ValueError("Choose service_account or adc.")

client = storage.Client(project=PROJECT_ID, credentials=credentials)
print("Selected project ID:", client.project)
print("Project supplied by credentials, if available:", credential_project)

## 8. Create a GCS bucket

Bucket names must be globally unique. The random suffix reduces name collisions without using a date. Choose a location allowed by your project; the example uses `us-central1`.

The bucket uses uniform bucket-level access, so access is managed through IAM. Run this creation cell once per exercise; rerunning it creates another bucket.

[Storage client methods](https://docs.cloud.google.com/python/docs/reference/storage/latest/google.cloud.storage.client.Client)

In [ ]:
from uuid import uuid4

BUCKET_NAME = f"gcs-python-demo-{uuid4().hex[:12]}"
LOCATION = "us-central1"

bucket = client.bucket(BUCKET_NAME)
bucket.iam_configuration.uniform_bucket_level_access_enabled = True
bucket = client.create_bucket(bucket, location=LOCATION)
print("Created bucket:", bucket.name)

## 9. List buckets in the project

`list_buckets()` returns buckets in the client's selected project.

In [ ]:
for item in client.list_buckets():
    print(item.name, item.location)

## 10. Create a small local file

Create a sample CSV beside the notebook. Local paths refer to the machine running the kernel.

In [ ]:
LOCAL_FILE = Path("gcs_demo_orders.csv")
LOCAL_FILE.write_text(
    """order_id,product,amount
1001,Keyboard,1500
1002,Mouse,700
""",
    encoding="utf-8",
)
print("Local file:", LOCAL_FILE.resolve())

## 11. Upload the file

A GCS **blob** represents an object. `raw/orders/gcs_demo_orders.csv` is the object name; its slashes act as a prefix for organizing files.

`if_generation_match=0` creates the object only if that name has no live object, avoiding an accidental overwrite. Rerunning this upload for an existing object produces a precondition error.

[Object methods](https://docs.cloud.google.com/python/docs/reference/storage/latest/google.cloud.storage.blob.Blob)

In [ ]:
OBJECT_NAME = "raw/orders/gcs_demo_orders.csv"
blob = bucket.blob(OBJECT_NAME)
blob.upload_from_filename(
    str(LOCAL_FILE),
    content_type="text/csv",
    if_generation_match=0,
)
uploaded_generation = blob.generation
print("Uploaded:", f"gs://{BUCKET_NAME}/{OBJECT_NAME}")

## 12. List files in the bucket

List every live object, then list only objects whose names begin with `raw/orders/`.

In [ ]:
for item in client.list_blobs(BUCKET_NAME):
    print(item.name, item.size, "bytes")

In [ ]:
for item in client.list_blobs(BUCKET_NAME, prefix="raw/orders/"):
    print(item.name)

## 13. Download the file

Save the uploaded object under a different local filename. The generation condition selects the version uploaded in this exercise.

In [ ]:
DOWNLOAD_FILE = Path("downloaded_gcs_demo_orders.csv")
blob.download_to_filename(
    str(DOWNLOAD_FILE),
    if_generation_match=uploaded_generation,
)
print("Downloaded to:", DOWNLOAD_FILE.resolve())

## 14. Delete the uploaded file from GCS

Run this cell when finished with the uploaded object. The generation condition prevents deleting a replacement uploaded by another process. The local files and bucket remain.

If the bucket has soft delete enabled, the deleted object may remain recoverable during its retention period.

In [ ]:
blob.delete(if_generation_match=uploaded_generation)
print("Deleted object:", f"gs://{BUCKET_NAME}/{OBJECT_NAME}")